# 04 — Evaluate and Listen

**What:** Run evaluation on a held-out real test set, render SI-SDR table,
side-by-side spectrograms + audio.
**Prerequisites:** Trained model + real test set.
**Runtime:** ~5 minutes.  **GPU needed:** Yes.

## 1. Run evaluation

In [ ]:
from pathlib import Path
import torch
from eval.evaluate import evaluate_dir

test_dir = Path('data/real_test')
if test_dir.exists():
    result = evaluate_dir(
        test_dir=test_dir, sig='<YOUR_SIG>',
        repo=Path('release_models'),
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    print(f"Mean SI-SDR: {result['mean_si_sdr']:.2f} dB")
else:
    print(f"Create {test_dir}/ with test mixtures + .flute.wav ground truth")

## 2. Spectrogram comparison

In [ ]:
import matplotlib.pyplot as plt
from data_pipeline.audio_utils import load_audio

def plot_spec(wav, sr=44100, ax=None, title=''):
    if ax is None:
        ax = plt.gca()
    wav_1d = wav.mean(dim=0) if wav.ndim > 1 else wav
    if wav_1d.numel() < 2048:
        ax.text(0.5, 0.5, 'Too short', ha='center')
        return
    D = torch.stft(wav_1d, n_fft=2048, hop_length=512,
                   window=torch.hann_window(2048), return_complex=True)
    ax.imshow(D.abs().log10().numpy(), aspect='auto', origin='lower', cmap='magma')
    ax.set_title(title)

sample = Path('data/real_test/example.wav')
if sample.exists():
    mix = load_audio(sample)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_spec(mix, ax=axes[0], title='Original mixture')
    axes[1].set_title('Extracted flute (run separation first)')
    plt.tight_layout()
    plt.show()

## 3. Listen

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

if sample.exists():
    data, sr = sf.read(str(sample))
    print('Original mixture:')
    display(Audio(data.T, rate=sr))